# Unit Tests
Run all backend connection-layer tests from this notebook.

In [ ]:
import json
import unittest

import backend.server as server
from backend.models import create_connection_state, create_user_session
from backend.protocol import ProtocolError, decode_message, encode_message


In [ ]:
class DummySocket:
    def __init__(self):
        self.sent = []

    def sendall(self, data):
        self.sent.append(data)

    def shutdown(self, how):
        return None

    def close(self):
        return None


def parse_sent_messages(sock):
    messages = []
    for raw in sock.sent:
        line = raw.decode('utf-8').strip()
        messages.append(json.loads(line))
    return messages


class TestProtocol(unittest.TestCase):
    def test_encode_message_has_newline_and_shape(self):
        raw = encode_message('LOGIN', {'username': 'Ali'})

        self.assertTrue(raw.endswith(b'\n'))
        parsed = json.loads(raw.decode('utf-8'))
        self.assertEqual(parsed['type'], 'LOGIN')
        self.assertEqual(parsed['payload'], {'username': 'Ali'})

    def test_decode_message_valid(self):
        line = b'{"type":"WAITING","payload":{}}'
        parsed = decode_message(line)
        self.assertEqual(parsed, {'type': 'WAITING', 'payload': {}})

    def test_decode_message_rejects_invalid_json(self):
        with self.assertRaises(ProtocolError):
            decode_message(b'{"type":"LOGIN"')

    def test_decode_message_rejects_missing_type(self):
        with self.assertRaises(ProtocolError):
            decode_message(b'{"payload":{}}')

    def test_decode_message_rejects_non_object_payload(self):
        with self.assertRaises(ProtocolError):
            decode_message(b'{"type":"LOGIN","payload":[]}')


class TestServerConnectionLayer(unittest.TestCase):
    def setUp(self):
        server.STATE = create_connection_state()
        server.RUNNING.set()

    def tearDown(self):
        server.RUNNING.clear()

    def test_handle_login_success_adds_user_and_broadcasts(self):
        sock = DummySocket()
        session = create_user_session(sock, ('127.0.0.1', 5001))

        server.handle_login(session, {'username': 'Ali'})

        self.assertEqual(session['username'], 'Ali')
        self.assertIn('Ali', server.STATE['online_users'])

        messages = parse_sent_messages(sock)
        self.assertEqual(messages[0]['type'], 'LOGIN_OK')
        self.assertEqual(messages[1]['type'], 'ONLINE_USERS')
        self.assertEqual(messages[1]['payload']['users'], ['Ali'])

    def test_handle_login_rejects_duplicate_username(self):
        first_sock = DummySocket()
        first_session = create_user_session(first_sock, ('127.0.0.1', 5001))
        server.handle_login(first_session, {'username': 'Ali'})

        second_sock = DummySocket()
        second_session = create_user_session(second_sock, ('127.0.0.1', 5002))
        server.handle_login(second_session, {'username': 'Ali'})

        self.assertIsNone(second_session['username'])
        messages = parse_sent_messages(second_sock)
        self.assertEqual(messages[0]['type'], 'LOGIN_REJECT')

    def test_dispatch_message_requires_login_first(self):
        sock = DummySocket()
        session = create_user_session(sock, ('127.0.0.1', 5001))

        server.dispatch_message(session, {'type': 'WAITING', 'payload': {}})

        messages = parse_sent_messages(sock)
        self.assertEqual(messages[0]['type'], 'ERROR')

    def test_waiting_and_spectator_transitions(self):
        sock = DummySocket()
        session = create_user_session(sock, ('127.0.0.1', 5001))
        session['username'] = 'Ali'
        server.STATE['online_users']['Ali'] = session

        server.dispatch_message(session, {'type': 'WAITING', 'payload': {}})
        self.assertIn('Ali', server.STATE['waiting_players'])
        self.assertNotIn('Ali', server.STATE['spectators'])

        server.dispatch_message(session, {'type': 'WATCH_MATCH', 'payload': {}})
        self.assertNotIn('Ali', server.STATE['waiting_players'])
        self.assertIn('Ali', server.STATE['spectators'])

        messages = parse_sent_messages(sock)
        self.assertEqual(messages[0]['type'], 'WAITING')
        self.assertEqual(messages[1]['type'], 'WATCH_MATCH')


In [ ]:
suite = unittest.TestSuite()
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestProtocol))
suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(TestServerConnectionLayer))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

if not result.wasSuccessful():
    raise AssertionError('Some unit tests failed')
